In [1]:
import numpy as np
import pandas as pd
import pymarc as pm

In [4]:
reader = pm.marcxml.parse_xml_to_array('data/BVB/b3kat_export_2025_05_teil02.xml')

In [3]:
# take back memory after done with preprocessing
del reader

In [ ]:
#wget, curl
#DOM, SAX based methods
# 020 = ISBN
# 041 = Language Code
# 100 = Author Name
# 240 = Title
# 245 Title Statement a, b, c
# 264a Place of production, publication, distribution, manufacture
# 264b Name of producer, publisher, distributor, manufacturer
# 264c Date of production, publication, distribution, manufacture, or copyright notice 
# 300a Number of Pages
# 336b Content Type
# 084 where $2 is rvk Classification number of rvk
# 655a where $2 is gnd or gnd-content Genre
# 650a where $2 is gnd Topical term or geographic name entry element

In [52]:
# TODO: figure out how to get unicode
def get_record_metadata(record, field_no, subfields, restraint=False):
    res = ''
    if not restraint:
        # look for match on field_no in record else return empty string
        field = record.get_fields(field_no) if record.get_fields(field_no) else None
        if not field:
            return res
        else:
            # iterate over all fields with matching field no only gets first subfield instance for now
            for sub in field:
                # add metadata information of subfield
                for i, subfield in enumerate(subfields):
                    # add space if multiple subfields
                    if i > 0 and res:
                        if sub.get_subfields(subfield):
                            res += ' ' + sub.get_subfields(subfield)[0]
                    else:
                        if sub.get_subfields(subfield):
                            res += sub.get_subfields(subfield)[0]
                break
        return res
    else:
        # restraint condition if looking for specific fields like rkv or gnd does not provide support for more than one subfield
        field = record.get_fields(field_no) if record.get_fields(field_no) else None
        if not field:
            return ''
        else:
            # iterate over all subfields to find restraint condition
            for sub in field:
                if sub.get_subfields('2'):
                    if sub.get_subfields('2')[0] == restraint[0] or sub.get_subfields('2')[0] == restraint[1]:
                        if res:
                            res +=  ', ' + sub.get_subfields(subfields[0])[0]
                        else:
                            res += sub.get_subfields(subfields[0])[0]
        return res

In [66]:
# make dictionary of MARC21 metadata fields to extract
fields = {
    'ISBN' : ['020', ['a'], False],
    'Language Code': ['041', ['a'], False],
    'Author': ['100', ['a'], False],
    'Title': ['240', ['a'], False],
    'Title Statement': ['245', ['a', 'b', 'c'], False],
    'Place': ['264', ['a'], False],
    'Publisher': ['264', ['b'], False],
    'Date' : ['264', ['c'], False],
    'Number of Pages': ['300', ['a'], False],
    'Content Type': ['336', ['b'], False], 
    'rvk Classification': ['084', ['a'], ['rvk', '']],
    'gnd Topical Term': ['650', ['a'], ['gnd', '']],
    'gnd Genre' : ['655', ['a'], ['gnd', 'gnd-content']]}
# make dictionary from reader using selected metadata fields to convert to dataframe
df_dict = {}
for key, field in fields.items():
    temp = []
    for record in reader:
        temp.append(get_record_metadata(record=record, field_no=field[0], subfields=field[1], restraint=field[2]))
    df_dict[key] = temp

In [67]:
df = pd.DataFrame.from_dict(df_dict)

In [70]:
df.to_csv('data/converted_data/b3kat_01', sep=';')

In [5]:
df = pd.read_csv('data/converted_data/b3kat_01', sep=';', dtype=str)

In [8]:
df.head(50)

,Unnamed: 0,ISBN,Language Code,Author,Title,Title Statement,Place,Publisher,Date,Number of Pages,Content Type,rvk Classification,gnd Topical Term,gnd Genre
0,0,NaN,eng,NaN,NaN,WRS-Steuer-und-Wirtschafts-Service / Sonderdruck,München,"Verl. Wirtschaft, Recht u. Steuern",1987,NaN,txt,NaN,NaN,NaN
1,1,NaN,ger,NaN,NaN,Goldmann Magnum,München,Goldmann,1987,NaN,txt,NaN,NaN,Monografische Reihe
2,2,NaN,ger,"Waghenaer, Lucas Janszoon",NaN,... Theil deß Spiegels der Seefart von Navigat...,Ambsterdam,Cornelius Claußsohn,1589,"36 S., 46 Doppelbl.",txt,NaN,NaN,NaN
3,3,NaN,und,NaN,NaN,Mémoire collective,Dunkerque,"Westhoeck, Ed. des Beffrois",NaN,NaN,txt,NaN,NaN,NaN
4,4,NaN,und,NaN,NaN,Ancient and medieval philosophy / Series 2 ...,Leuven u.a.,Univ. Pr. u.a.,NaN,NaN,txt,NaN,NaN,NaN
5,5,NaN,und,NaN,NaN,BLV-Bestimmungsbuch,München u.a.,BLV-Verl.-Ges.,NaN,NaN,txt,NaN,NaN,NaN
6,6,3463008386,ger,"Kunze, Michael",NaN,Straße ins Feuer vom Leben und Sterben in der ...,München,Kindler,1982,397 S.,txt,"GN 9999, LC 41005, NS 4470","Strafrecht, Hexenprozess",Fiktionale Darstellung
7,7,NaN,ger,"Leppmann, Wolfgang",NaN,Goethe und die Deutschen d. Nachruhm e. Dichte...,Bern ; München,Scherz,1982,318 S.,txt,"GK 4291, GK 4351",Rezeption,Biografie
8,8,3434004246,ger,"Fromm, Erich",The sane society,Wege aus einer kranken Gesellschaft,Frankfurt am Main,Europ. Verl.-Anst.,1982,354 S.,txt,"CU 2567, MS 1290","Gesellschaft, Automation, Humanität, Sozialer ...",NaN
9,9,3570028364,ger,"Vorkort, Walter",NaN,Wie die weißen Engel die blauen Tiger zur Schn...,München,Bertelsmann,1982,123 S.,txt,NaN,NaN,NaN


In [16]:
df['gnd Topical Term'].value_counts()

gnd Topical Term
Geschichte                                                      7872
Vedute                                                          3319
Karte                                                           2212
Befestigung                                                     2121
Biografie                                                       1686
                                                                ... 
Militärchirurgie, Militärmedizin                                   1
Elektronik, Deutsch, Elektrotechnik, Englisch, Wörterbuch          1
Arzt, Psychische Störung, Patient, Somatisierung, Prävention       1
Interpretation, Hermeneutik                                        1
Parlament, Gesandtschaft, Französisch, Asylrecht                   1
Name: count, Length: 287627, dtype: int64

In [9]:
len(df)

942285